# Research 74 — retrained tuned CatBoost + LightGBM reference on revised Task 2 data

The prior OOF results were generated from a superseded train/test snapshot and are not comparable/blendable with this run. This notebook reuses the historical selected recipe as-is (fixed parameters, no Optuna rerun), performs three-fold stratified OOF evaluation on the revised training CSV, then refits both models on all labeled rows for prediction-only inference on the revised test CSV.

Exact SHA-256 guards prevent training on a different competition revision. Test rows are not loaded until OOF metrics are computed. The Kaggle kernel may have a T4 enabled, but this baseline deliberately keeps the original CPU model backends/parameters for a faithful recipe reproduction. Outputs stay keyed by `player_id`; this notebook does not edit `submit/index.csv`.

In [1]:
import hashlib
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

REVISION = "2026-11-12"
SEED = 42
EXPECTED_TRAIN_SHA256 = "0cdaa5fad6c17a7f27d965b988cd72f6113a1a53c1f88bcf5d12c25128249f48"
EXPECTED_TEST_SHA256 = "224345827c690d4383fa1c81d4a7baecfb22735ed80189b1d90504dfcfa9b703"


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_verified_train():
    roots = [
        Path("/kaggle/input"),
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parents[1] / "data",
    ]
    for root in roots:
        if not root.exists():
            continue
        candidates = (
            [root / "train.csv"] if (root / "train.csv").is_file() else root.rglob("train.csv")
        )
        for train_path in candidates:
            if sha256(train_path) == EXPECTED_TRAIN_SHA256:
                return train_path.parent
    raise FileNotFoundError(
        "No attached train.csv matches the revised data SHA-256; aborting before fitting."
    )


DATA_DIR = locate_verified_train()
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
assert sha256(TRAIN_PATH) == EXPECTED_TRAIN_SHA256, "Wrong train revision."
assert TEST_PATH.is_file(), f"Missing paired test.csv under {DATA_DIR}"
if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working")
    MODEL_DIR = OUTPUT_DIR
else:
    OUTPUT_DIR = DATA_DIR.parent / "predictions"
    MODEL_DIR = DATA_DIR.parent / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(TRAIN_PATH)
assert train.shape == (104_000, 46), train.shape
assert "segment" in train and "player_id" in train
assert train["player_id"].notna().all() and train["player_id"].is_unique
y = train["segment"].to_numpy(dtype=int)
assert np.array_equal(np.unique(y), np.arange(4)), np.unique(y)
print(
    {
        "revision": REVISION,
        "train_sha256": sha256(TRAIN_PATH),
        "train_shape": train.shape,
        "class_counts": train["segment"].value_counts().sort_index().to_dict(),
        "test_loaded": False,
    }
)

{'revision': '2026-11-12', 'train_sha256': '0cdaa5fad6c17a7f27d965b988cd72f6113a1a53c1f88bcf5d12c25128249f48', 'train_shape': (104000, 46), 'class_counts': {0: 40957, 1: 26032, 2: 21029, 3: 15982}, 'test_loaded': False}


In [2]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()
raw = train.drop(columns=["segment", "player_id"]).copy()
raw_categories = raw.select_dtypes(include=["object", "string"]).columns.tolist()
raw[raw_categories] = raw[raw_categories].fillna("__MISSING__").astype(str)

LGB_PARAMS = {
    "learning_rate": 0.063863707925569,
    "num_leaves": 59,
    "max_depth": 4,
    "min_child_samples": 81,
    "reg_lambda": 0.12942956115511214,
    "colsample_bytree": 0.7975990992289793,
}
LGB_ITERATIONS = 440


def make_lgb():
    return LGBMClassifier(
        n_estimators=LGB_ITERATIONS,
        objective="multiclass",
        random_state=SEED,
        n_jobs=4,
        verbosity=-1,
        **LGB_PARAMS,
    )


def encode_pair(fit_frame, pred_frame):
    fit_frame, pred_frame = fit_frame.copy(), pred_frame.copy()
    for column in categorical:
        categories = sorted(fit_frame[column].dropna().unique())
        dtype = pd.CategoricalDtype(categories=categories)
        fit_frame[column] = fit_frame[column].astype(dtype)
        pred_frame[column] = pred_frame[column].astype(dtype)
    return fit_frame, pred_frame


def make_cat():
    return CatBoostClassifier(
        iterations=300,
        depth=7,
        learning_rate=0.08,
        loss_function="MultiClass",
        random_seed=SEED,
        thread_count=4,
        allow_writing_files=False,
        verbose=False,
    )


def ordered_proba(model, x):
    probabilities = np.asarray(model.predict_proba(x), dtype=np.float64)
    classes = np.asarray(model.classes_, dtype=int)
    assert set(classes.tolist()) == {0, 1, 2, 3}, classes
    probabilities = probabilities[:, np.argsort(classes)]
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
    return probabilities


print(
    {
        "engineered_features": X.shape[1],
        "lgb_categorical_columns": len(categorical),
        "catboost_categorical_columns": len(raw_categories),
        "fixed_lightgbm_params": LGB_PARAMS,
        "lightgbm_iterations": LGB_ITERATIONS,
        "catboost": {"iterations": 300, "depth": 7, "learning_rate": 0.08},
    }
)

{'engineered_features': 51, 'lgb_categorical_columns': 10, 'catboost_categorical_columns': 10, 'fixed_lightgbm_params': {'learning_rate': 0.063863707925569, 'num_leaves': 59, 'max_depth': 4, 'min_child_samples': 81, 'reg_lambda': 0.12942956115511214, 'colsample_bytree': 0.7975990992289793}, 'lightgbm_iterations': 440, 'catboost': {'iterations': 300, 'depth': 7, 'learning_rate': 0.08}}


In [3]:
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_lgb = np.full((len(train), 4), np.nan, dtype=np.float64)
oof_cat = np.full((len(train), 4), np.nan, dtype=np.float64)
oof_fold = np.zeros(len(train), dtype=np.int8)
fold_rows = []
started = time.perf_counter()
for fold_number, (fit_idx, valid_idx) in enumerate(folds.split(X, y), start=1):
    x_fit, x_valid = encode_pair(X.iloc[fit_idx], X.iloc[valid_idx])
    lgb = make_lgb()
    lgb.fit(x_fit, y[fit_idx])
    oof_lgb[valid_idx] = ordered_proba(lgb, x_valid)
    cat = make_cat()
    cat.fit(raw.iloc[fit_idx], y[fit_idx], cat_features=raw_categories)
    oof_cat[valid_idx] = ordered_proba(cat, raw.iloc[valid_idx])
    oof_fold[valid_idx] = fold_number
    for model_name, probabilities in [
        ("lightgbm", oof_lgb),
        ("catboost", oof_cat),
        ("equal_blend", (oof_lgb + oof_cat) / 2),
    ]:
        prediction = probabilities[valid_idx].argmax(axis=1)
        fold_rows.append(
            {
                "fold": fold_number,
                "model": model_name,
                "macro_f1": f1_score(y[valid_idx], prediction, average="macro"),
                "weighted_f1": f1_score(y[valid_idx], prediction, average="weighted"),
            }
        )
    print(
        f"Fold {fold_number}/3 finished ({len(fit_idx):,} train, {len(valid_idx):,} validation)",
        flush=True,
    )
assert (oof_fold > 0).all()
assert np.isfinite(oof_lgb).all() and np.isfinite(oof_cat).all()
elapsed_seconds = time.perf_counter() - started
oof_blend = (oof_lgb + oof_cat) / 2
score_rows = []
for model_name, probabilities in [
    ("catboost", oof_cat),
    ("lightgbm", oof_lgb),
    ("equal_blend", oof_blend),
]:
    prediction = probabilities.argmax(axis=1)
    score_rows.append(
        {
            "model": model_name,
            "macro_f1": f1_score(y, prediction, average="macro"),
            "weighted_f1": f1_score(y, prediction, average="weighted"),
        }
    )
scores = pd.DataFrame(score_rows).sort_values("macro_f1", ascending=False)
fold_scores = pd.DataFrame(fold_rows)
display(scores)
display(fold_scores)
print(f"OOF duration: {elapsed_seconds:.1f}s. Test rows have not been loaded.")

Fold 1/3 finished (69,333 train, 34,667 validation)


Fold 2/3 finished (69,333 train, 34,667 validation)


Fold 3/3 finished (69,334 train, 34,666 validation)


,model,macro_f1,weighted_f1
2,equal_blend,0.751314,0.764749
1,lightgbm,0.748673,0.762190
0,catboost,0.747610,0.761392


,fold,model,macro_f1,weighted_f1
0,1,lightgbm,0.747065,0.760809
1,1,catboost,0.746072,0.759873
2,1,equal_blend,0.749884,0.763637
3,2,lightgbm,0.747557,0.761286
4,2,catboost,0.748059,0.761896
5,2,equal_blend,0.750850,0.764260
6,3,lightgbm,0.751389,0.764473
7,3,catboost,0.748692,0.762403
8,3,equal_blend,0.753198,0.766346


OOF duration: 812.2s. Test rows have not been loaded.


In [4]:
score_tag = f"macro-f1-{scores.iloc[0].macro_f1:.6f}"
prefix = f"segment-tuned-catboost-lightgbm-revised-{REVISION}-{score_tag}"
oof_frame = pd.DataFrame(
    {"player_id": train["player_id"].astype(str), "segment": y, "fold": oof_fold}
)
for i in range(4):
    oof_frame[f"catboost_prob_{i}"] = oof_cat[:, i]
    oof_frame[f"lightgbm_prob_{i}"] = oof_lgb[:, i]
    oof_frame[f"blend_prob_{i}"] = oof_blend[:, i]
oof_frame["predicted_segment"] = oof_blend.argmax(axis=1)
assert oof_frame["player_id"].equals(train["player_id"].astype(str))
oof_path = OUTPUT_DIR / f"{prefix}-oof.csv"
oof_frame.to_csv(oof_path, index=False)
scores.to_csv(OUTPUT_DIR / f"{prefix}-scores.csv", index=False)
fold_scores.to_csv(OUTPUT_DIR / f"{prefix}-fold-scores.csv", index=False)
(OUTPUT_DIR / f"{prefix}-oof-report.json").write_text(
    json.dumps(
        {
            "revision": REVISION,
            "train_sha256": EXPECTED_TRAIN_SHA256,
            "train_rows": len(train),
            "folds": 3,
            "cv_seed": SEED,
            "oof_seconds": round(elapsed_seconds, 2),
            "scores": scores.to_dict(orient="records"),
            "fold_scores": fold_rows,
            "classification_report_equal_blend": classification_report(
                y, oof_blend.argmax(axis=1), output_dict=True, zero_division=0
            ),
            "confusion_matrix_equal_blend": confusion_matrix(
                y, oof_blend.argmax(axis=1), labels=[0, 1, 2, 3]
            ).tolist(),
            "lightgbm_params": LGB_PARAMS,
            "lightgbm_iterations": LGB_ITERATIONS,
            "catboost_params": {
                "iterations": 300,
                "depth": 7,
                "learning_rate": 0.08,
                "thread_count": 4,
            },
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Saved OOF and score artifacts under {OUTPUT_DIR}")

assert sha256(TEST_PATH) == EXPECTED_TEST_SHA256, (
    "Wrong test revision; stopping before test inference."
)
test = pd.read_csv(TEST_PATH)
assert test.shape == (25_889, 45), test.shape
assert "segment" not in test and test["player_id"].notna().all() and test["player_id"].is_unique
assert list(train.drop(columns="segment").columns) == list(test.columns)
assert set(train["player_id"].astype(str)).isdisjoint(test["player_id"].astype(str))
print({"test_sha256": sha256(TEST_PATH), "test_rows": len(test), "prediction_only": True})

X_full, X_test = encode_pair(X, features(test))
final_lgb = make_lgb().fit(X_full, y)
test_lgb = ordered_proba(final_lgb, X_test)
test_raw = test[raw.columns].copy()
test_raw[raw_categories] = test_raw[raw_categories].fillna("__MISSING__").astype(str)
final_cat = make_cat().fit(raw, y, cat_features=raw_categories)
test_cat = ordered_proba(final_cat, test_raw)
test_blend = (test_cat + test_lgb) / 2
test_frame = pd.DataFrame({"player_id": test["player_id"].astype(str)})
for i in range(4):
    test_frame[f"catboost_prob_{i}"] = test_cat[:, i]
    test_frame[f"lightgbm_prob_{i}"] = test_lgb[:, i]
    test_frame[f"blend_prob_{i}"] = test_blend[:, i]
test_frame["predicted_segment"] = test_blend.argmax(axis=1)
assert test_frame["player_id"].equals(test["player_id"].astype(str))
assert test_frame["predicted_segment"].isin([0, 1, 2, 3]).all()
test_path = OUTPUT_DIR / f"{prefix}-test-predictions.csv"
test_frame.to_csv(test_path, index=False)
final_cat.save_model(str(MODEL_DIR / f"{prefix}-catboost.cbm"))
joblib.dump(final_lgb, MODEL_DIR / f"{prefix}-lightgbm.joblib")
display(test_frame.head())
print(
    {
        "saved_test_predictions": str(test_path),
        "rows": len(test_frame),
        "class_distribution": test_frame["predicted_segment"]
        .value_counts(normalize=True)
        .sort_index()
        .to_dict(),
        "submit_index_edited": False,
    }
)

Saved OOF and score artifacts under C:\Users\Gunte\Workspace\Workspace\works\CPE\ml00x\task2\notebook\tried\predictions


{'test_sha256': '224345827c690d4383fa1c81d4a7baecfb22735ed80189b1d90504dfcfa9b703', 'test_rows': 25889, 'prediction_only': True}


,player_id,catboost_prob_0,lightgbm_prob_0,blend_prob_0,catboost_prob_1,lightgbm_prob_1,blend_prob_1,catboost_prob_2,lightgbm_prob_2,blend_prob_2,catboost_prob_3,lightgbm_prob_3,blend_prob_3,predicted_segment
0,P090473,0.003666,0.002178,0.002922,0.016640,0.014190,0.015415,0.064386,0.042203,0.053295,0.915309,0.941429,0.928369,3
1,P061310,0.009839,0.012315,0.011077,0.006460,0.014982,0.010721,0.505075,0.575473,0.540274,0.478626,0.397230,0.437928,2
2,P077253,0.009492,0.017356,0.013424,0.002089,0.001543,0.001816,0.966340,0.954348,0.960344,0.022079,0.026753,0.024416,2
3,P054734,0.133257,0.056069,0.094663,0.849514,0.929943,0.889729,0.006929,0.004620,0.005775,0.010300,0.009369,0.009834,1
4,P076202,0.813514,0.876840,0.845177,0.167582,0.108786,0.138184,0.010377,0.008068,0.009223,0.008527,0.006306,0.007417,0


{'saved_test_predictions': 'C:\\Users\\Gunte\\Workspace\\Workspace\\works\\CPE\\ml00x\\task2\\notebook\\tried\\predictions\\segment-tuned-catboost-lightgbm-revised-2026-11-12-macro-f1-0.751314-test-predictions.csv', 'rows': 25889, 'class_distribution': {0: 0.4328865541349608, 1: 0.24608907257908763, 2: 0.18386187183745992, 3: 0.13716250144849162}, 'submit_index_edited': False}
